## COVID Drivers: Modeling

This notebook models
POST_COVID ~ NHTSA_AGG_DRIVING

### Table of Contents
* [Read the Data](#read)</BR>
* [Preprocessing](#prep)</BR>
* [XGBoost](#xgb)</BR>
* [XGBoost with GridSearchCV](#xgb-gs)</BR>
* [Review Models](#review)


Import packages

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

import xgboost as xgb
from functools import reduce
#import prince

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path_in = '/content/drive/MyDrive/Colab Notebooks/Case Studies in Data Science/data/ready/ready_data_final.csv'

In [ ]:
path_out = 'metrics_07_prim_modl_xgb.csv'

### <a id='read'>Read the data</a>

In [ ]:
df_init = pd.read_csv(path_in, low_memory=False)

In [ ]:
df_init['CRASH_DATE'] = pd.to_datetime(df_init['CRASH_DATE'])

In [ ]:
df = df_init.set_index('CRASH_DATE').drop(columns=['CRN']).copy()

In [ ]:
model_metrics = []

In [ ]:
df.columns.tolist()

['POST_COVID',
 'ALCOHOL_RELATED',
 'CELL_PHONE',
 'DISTRACTED',
 'DRINKING_DRIVER',
 'DRIVER_16YR',
 'DRIVER_17YR',
 'DRIVER_18YR',
 'DRIVER_19YR',
 'DRIVER_20YR',
 'DRIVER_50_64YR',
 'DRIVER_65_74YR',
 'DRIVER_75PLUS',
 'DRUGGED_DRIVER',
 'DRUG_RELATED',
 'FATIGUE_ASLEEP',
 'HIT_RUN',
 'ILLEGAL_DRUG_RELATED',
 'IMPAIRED_DRIVER',
 'IMPAIRED_NONMOTORIST',
 'MARIJUANA_DRUGGED_DRIVER',
 'MARIJUANA_RELATED',
 'MATURE_DRIVER',
 'MC_DRINKING_DRIVER',
 'OPIOID_RELATED',
 'UNDERAGE_DRNK_DRV',
 'UNLICENSED',
 'YOUNG_DRIVER',
 'AGGRESSIVE_DRIVING',
 'NHTSA_AGG_DRIVING',
 'NO_CLEARANCE',
 'RUNNING_RED_LT',
 'RUNNING_STOP_SIGN',
 'SPEEDING',
 'SPEEDING_RELATED',
 'TAILGATING',
 'COUNTYx',
 'URBAN_RURALx']

### <a id='prep'>Preprocessing</a>

In [ ]:
X = df.loc[:,['POST_COVID']].copy()

In [ ]:
y = df['NHTSA_AGG_DRIVING']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### <a id='xgb'>XGBoost</a>

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('xgboost', xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        objective='binary:logistic'
     ))
])

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = xgb_pipeline.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Predicted probabilities for the class 1
y_pred_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Precision: tp / (tp + fp)
precision = precision_score(y_test, y_pred)

In [ ]:
# Recall: tp / (tp + fn)
recall = recall_score(y_test, y_pred)

In [ ]:
# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

In [ ]:
# ROC AUC: Area Under the Receiver Operating Characteristic Curve
roc_auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
measure_titles = ['Accuracy','Precision','Recall','F1 Score','ROC AUC']
measure_value = [accuracy,precision,recall,f1,roc_auc]
model = 'XGBoost'

In [ ]:
aggdrv_xgb = pd.DataFrame({'Measure':measure_titles,
                            model:measure_value})

In [ ]:
aggdrv_xgb['XGBoost'] = [round(x, 4) for x in aggdrv_xgb['XGBoost']]

In [ ]:
aggdrv_xgb

In [ ]:
model_metrics.append(aggdrv_xgb)

### <a id='xgb-gs'>XGBoost with GridSearchCV</a>

In [ ]:
# Number of trees in random forest
n_estimators = [100, 1000]
# Maximum number of levels in tree
max_depth = [None, 5, 10]
# Step size at each boosting iteration
learning_rate = [0.01, 0.2]
# Minimum sum of instance weight (hessian) required in a child node
min_child_weight = [0, 1]
# Minimum loss reduction required to make a further partition on a leaf node of the tree - gamma
min_split_loss = [0, 0.1]
# Method of selecting samples for training each tree
subsample = [0.5, 1]
# L2 regularization term on weights
reg_lambda = [1, 2]
# L1 regularization term on weights
reg_alpha = [0, 1]

# Create the random grid
param_grid = {'clf__max_depth': max_depth,
               'clf__learning_rate': learning_rate,
               'clf__min_child_weight': min_child_weight,
               'clf__min_split_loss': min_split_loss,
               'clf__subsample': subsample,
               'clf__reg_lambda': reg_lambda,
               'clf__reg_alpha': reg_alpha}

In [ ]:
clf_pipeline = Pipeline([
    ('clf', xgb.XGBClassifier(n_estimators=1000, early_stopping_rounds=10, random_state=42))
])

In [ ]:
grid_search = GridSearchCV(clf_pipeline,
                            param_grid=param_grid,
                            cv=5,
                            scoring='f1',
                            return_train_score=True,
                            refit=True,
                            verbose=1)

In [ ]:
grid_search.fit(
    X_train, y_train,
    clf__eval_set=[(X_test, y_test)]
)

Streaming output truncated to the last 5000 lines.
[467]	validation_0-logloss:0.68597
[468]	validation_0-logloss:0.68597
[469]	validation_0-logloss:0.68597
[470]	validation_0-logloss:0.68597
[471]	validation_0-logloss:0.68597
[472]	validation_0-logloss:0.68597
[473]	validation_0-logloss:0.68597
[474]	validation_0-logloss:0.68597
[475]	validation_0-logloss:0.68597
[476]	validation_0-logloss:0.68597
[477]	validation_0-logloss:0.68597
[478]	validation_0-logloss:0.68597
[479]	validation_0-logloss:0.68597
[480]	validation_0-logloss:0.68597
[481]	validation_0-logloss:0.68597
[482]	validation_0-logloss:0.68597
[483]	validation_0-logloss:0.68597
[484]	validation_0-logloss:0.68597
[485]	validation_0-logloss:0.68597
[486]	validation_0-logloss:0.68597
[487]	validation_0-logloss:0.68597
[488]	validation_0-logloss:0.68597
[489]	validation_0-logloss:0.68597
[490]	validation_0-logloss:0.68597
[491]	validation_0-logloss:0.68597
[492]	validation_0-logloss:0.68597
[493]	validation_0-logloss:0.68597
[494

In [ ]:
y_pred = grid_search.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Predicted probabilities for the class 1
y_pred_proba = grid_search.predict_proba(X_test)[:, 1]

In [ ]:
# Precision: tp / (tp + fp)
precision = precision_score(y_test, y_pred)

In [ ]:
# Recall: tp / (tp + fn)
recall = recall_score(y_test, y_pred)

In [ ]:
# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

In [ ]:
# ROC AUC: Area Under the Receiver Operating Characteristic Curve
roc_auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
measure_titles = ['Accuracy','Precision','Recall','F1 Score','ROC AUC']
measure_value = [accuracy,precision,recall,f1,roc_auc]
model = 'XGBoost_GridSearchCV'

In [ ]:
aggdrv_xgbgs = pd.DataFrame({'Measure':measure_titles,
                            model:measure_value})

In [ ]:
aggdrv_xgbgs['XGBoost_GridSearchCV'] = [round(x, 4) for x in aggdrv_xgbgs['XGBoost_GridSearchCV']]

In [ ]:
aggdrv_xgbgs

In [ ]:
model_metrics.append(aggdrv_xgbgs)

### <a id='review'>Review Models</a>

In [ ]:
merged_metrics = reduce(lambda left, right: pd.merge(left, right, on='Measure', how='inner'), model_metrics)

In [ ]:
merged_metrics

In [ ]:
merged_metrics.to_csv(path_out, index=False)